In [1]:
import requests
import pandas as pd

url = "https://www.fpbase.org/api/proteins/?format=json"
data = requests.get(url).json()

In [2]:
import requests
import pandas as pd

FPBASE_URL = "https://www.fpbase.org/graphql/"


def run_query(query):
    r = requests.post(
        FPBASE_URL,
        json={"query": query},
        headers={"Content-Type": "application/json"},
    )

    if r.status_code != 200:
        print(r.text)
        r.raise_for_status()

    data = r.json()

    if "errors" in data:
        raise Exception(data["errors"])

    return data["data"]


query = """
{
  proteins {
    name
    slug
    seq
    agg
    pdb
    cofactor

    defaultState {
      exMax
      emMax
      extCoeff
      qy
      brightness
    }
  }
}
"""

data = run_query(query)

rows = []

for p in data["proteins"]:
    st = p.get("defaultState") or {}

    rows.append({
        "name": p.get("name"),
        "FPBase_ID": p.get("slug"),
        "sequence": p.get("seq"),
        "aggregation_type": p.get("agg"),

        "excitation_wavelength": st.get("exMax"),
        "emission_wavelength": st.get("emMax"),
        "extinction_coefficient": st.get("extCoeff"),
        "quantum_yield": st.get("qy"),
        "brightness": st.get("brightness"),

        "pdb_codes": p.get("pdb"),
        "cofactor": p.get("cofactor"),
    })

df = pd.DataFrame(rows)

print(df.head())
print(df.shape)

            name    FPBase_ID  \
0            10B          10b   
1             11           11   
2            22G          22g   
3  (3-F)Tyr-EGFP  3-ftyr-egfp   
4             5B           5b   

                                            sequence aggregation_type  \
0  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...              NaN   
1  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...              NaN   
2  MSVIKPDMKIKLRMEGAVNGHPFAIEGVGLGKPFEGKQSMDLKVKE...                T   
3  SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...              NaN   
4  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...              NaN   

   excitation_wavelength  emission_wavelength  extinction_coefficient  \
0                  513.0                525.0                 30800.0   
1                  502.0                512.0                 33000.0   
2                    NaN                  NaN                     NaN   
3                  484.0                514.0                     NaN 

In [3]:
## Cleaning the dataseet

df_clean = df.dropna(subset=[
    "brightness",
    "quantum_yield",
    "extinction_coefficient",
    "sequence",
    "emission_wavelength"
])
original_count = len(df)
clean_count = len(df_clean)
removed_count = original_count - clean_count

print("Original:", original_count)
print("Remaining:", clean_count)
print("Removed:", removed_count)
df_clean.isna().sum()

Original: 1040
Remaining: 592
Removed: 448


name                        0
FPBase_ID                   0
sequence                    0
aggregation_type           72
excitation_wavelength       0
emission_wavelength         0
extinction_coefficient      0
quantum_yield               0
brightness                  0
pdb_codes                   0
cofactor                  541
dtype: int64

In [49]:
df[df["FPBase_ID"].str.contains("philov21", case=False, na=False)]


,name,FPBase_ID,sequence,aggregation_type,excitation_wavelength,emission_wavelength,extinction_coefficient,quantum_yield,brightness,pdb_codes,cofactor
794,phiLOV2.1,philov21,MEKSFVITDPRLPDYPIIFASDGFLELTEYSREEIMGRNARFLQGP...,M,451.0,501.0,13500.0,0.2,2.7,[],FL


In [ ]:
import pandas as pd
import openpyxl
df = df_clean.copy()

# --- Rename columns to desired labels ---
df = df.rename(columns={
    "name": "Name",
    "pdb_codes": "PDB code",
    "cofactor": "chromophore/ligand",
    "sequence": "Protein sequence",
    "excitation_wavelength": "Excitation wavelength",
    "emission_wavelength": "Emission wavelength",
    "extinction_coefficient": "EC value",
    "quantum_yield": "QY value",
    "brightness": "Brightness"
})

# --- Calculate Stokes shift ---
df["Stokes shift"] = df["Emission wavelength"] - df["Excitation wavelength"]

# --- Calculate kDa from sequence (approx 110 Da per amino acid) ---
df["kDa"] = df["Protein sequence"].str.len() * 110 / 1000

# --- Define desired column order ---
desired_order = [
    "Name",
    "PDB code",
    "chromophore/ligand",
    "Protein sequence",
    "Excitation wavelength",
    "Emission wavelength",
    "Stokes shift",
    "EC value",
    "QY value",
    "Brightness",
    "kDa"
]

# --- Add remaining columns at the end ---
remaining_cols = [col for col in df.columns if col not in desired_order]
df = df[desired_order + remaining_cols]

# --- Filter emission wavelength between 529 and 600 ---
filtered_df = df[
    (df["Emission wavelength"] > 300) &
    (df["Emission wavelength"] < 500)
]

# --- Save to Excel ---
filtered_df.to_excel("filtered_fluorescent_proteins_amelie.xlsx", index=False)

print("Saved:", filtered_df.shape[0], "rows")

Saved: 130 rows


In [ ]:
missing_pdb = df_clean[df_clean["pdb_codes"].apply(lambda x: len(x) == 0 if isinstance(x, list) else True)]

In [51]:
import requests
import time

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"


def search_pdb_by_sequence(seq):
    """
    Query RCSB sequence search API for closest PDB match.
    Returns top PDB ID or empty list.
    """

    query = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": 0.001,
                "identity_cutoff": 0.3,
                "sequence_type": "protein",
                "value": seq
            }
        },
        "request_options": {
            "pager": {
                "start": 0,
                "rows": 1
            }
        },
        "return_type": "entry"
    }

    try:
        r = requests.post(RCSB_SEARCH_URL, json=query, timeout=20)

        if r.status_code != 200:
            return []

        data = r.json()

        results = data.get("result_set", [])

        if not results:
            return []

        return [results[0]["identifier"]]

    except Exception:
        return []


def fill_pdb_sequence(row):
    if isinstance(row["pdb_codes"], list) and len(row["pdb_codes"]) > 0:
        return row["pdb_codes"]

    seq = row.get("sequence")

    if not seq or len(seq) < 50:
        return []

    return search_pdb_by_sequence(seq)


# ONLY run on missing subset (IMPORTANT — this is slow)
missing_idx = df_clean.index[
    df_clean["pdb_codes"].apply(lambda x: len(x) == 0 if isinstance(x, list) else True)
]

for i in missing_idx[:50]:  # start small
    df_clean.at[i, "pdb_codes_filled"] = fill_pdb_sequence(df_clean.loc[i])
    time.sleep(0.2)  # avoid rate limits

In [52]:
df_clean["pdb_codes_filled"].apply(lambda x: len(x) if isinstance(x, list) else 0).value_counts()
df_clean[df_clean["pdb_codes_filled"].apply(lambda x: isinstance(x, list) and len(x) > 0)].head()

,name,FPBase_ID,sequence,aggregation_type,excitation_wavelength,emission_wavelength,extinction_coefficient,quantum_yield,brightness,pdb_codes,cofactor,pdb_codes_filled
17,aceGFP,acegfp,MSKGAELFTGIVPILIELNGDVNGHKFSVSGEGEGDATYGKLTLKF...,M,480.0,505.0,50000.0,0.550,27.50,[3LVA],NaN,[3LVA]
34,amFP486,amfp486,MALSNKFIGDDMKMTYHMDGCVNGHYFTVKGEGNGKPYEGTQTSTF...,T,458.0,486.0,40000.0,0.240,9.60,[2A46],NaN,[2A46]
68,asulCP,asulcp,MASFLKKTMPFKTTIEGTVNGHYFKCTGKGEGNPFEGTQEMKIEVI...,NaN,572.0,595.0,120000.0,0.001,0.12,[1XQM],NaN,[1XQM]
71,avGFP,avgfp,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,D,395.0,509.0,25000.0,0.790,19.75,[1GFL],NaN,[1GFL]
84,AzamiRed1.0,azamired10,MVSVIKEEMKIKLRMEGTVNGHNFVIEGEGKGNPYEGTQTMDLKVT...,T,571.0,606.0,34100.0,0.650,22.16,[8I4K],NaN,[8I4K]


In [61]:
targets = [
    "phiLOV 2.1",
    "miniSOG",
    "mEos2-A69T",
    "Padron2",
    "rsEGFP",
    "Padron0.9",
    "aceGFP",
    "BrUSLEE",
    "rsGreen1",
    "Dreiklang",
    "Kohinoor",
    "WasCFP",
    "NowGFP",
    "mEosFP",
    "Thermostable Green Protein",
    "ppluGFP2",
    "muGFP",
    "mEos4b-L93M",
    "pcStar"
]

subset = df_clean[df_clean["name"].isin(targets)]

display(subset)

,name,FPBase_ID,sequence,aggregation_type,excitation_wavelength,emission_wavelength,extinction_coefficient,quantum_yield,brightness,pdb_codes,cofactor,pdb_codes_filled,name_norm
17,aceGFP,acegfp,MSKGAELFTGIVPILIELNGDVNGHKFSVSGEGEGDATYGKLTLKF...,M,480.0,505.0,50000.0,0.55,27.50,[3LVA],NaN,[3LVA],acegfp
100,BrUSLEE,bruslee,MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLK...,M,487.0,509.0,86000.0,0.30,25.80,[],NaN,[],bruslee
186,Dreiklang,dreiklang,MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLK...,M,511.0,529.0,83000.0,0.41,34.03,"[3ST3, 3ST2, 3ST4]",NaN,"[3ST3, 3ST2, 3ST4]",dreiklang
443,Kohinoor,kohinoor,MSVIKPDMKIKLRMEGAVNGHPFAIEGVGLGKPFEGKQSMDLKVKE...,M,495.0,514.0,62900.0,0.71,44.66,[],NaN,[],kohinoor
538,mEos2-A69T,meos2-a69t,MSAIKPDMKIKLRMEGNVNGHHFVIDGDGTGKPFEGKQSMDLEVKE...,M,565.0,580.0,11500.0,0.66,7.59,[5DTL],NaN,[5DTL],meos2-a69t
544,mEos4b-L93M,meos4b-l93m,MVSAIKPDMRIKLRMEGNVNGHHFVIDGDGTGKPYEGKQTMDLEVK...,M,569.0,580.0,57200.0,0.63,36.04,[9GVR],NaN,[9GVR],meos4b-l93m
548,mEosFP,meosfp,MSAIKPDMKINLRMEGNVNGHHFVIDGDGTGKPFEGKQSMDLEVKE...,M,569.0,581.0,37000.0,0.62,22.94,[3P8U],NaN,[3P8U],meosfp
579,miniSOG,minisog,MEKSFVITDPRLPDNPIIFASDGFLELTEYSREEILGRNGRFLQGP...,M,447.0,501.0,14100.0,0.43,6.06,[6GPU],FL,[6GPU],minisog
730,muGFP,mugfp,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,M,490.0,508.0,117000.0,0.45,52.65,[5JZL],NaN,[5JZL],mugfp
748,NowGFP,nowgfp,MVSKGEKLFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKMSLK...,M,494.0,502.0,56700.0,0.76,43.09,"[4rtc, 4rys, 4ryw]",NaN,"[4rtc, 4rys, 4ryw]",nowgfp
